In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, root_mean_squared_error

dataset = pd.read_csv("student_scores.csv")
dataset.head(15)

,SoGioHoc_Tuan,DiemChuyenCan,SoBuoiVang,SoGioChoiGame_Tuan,ChieuCao_cm,DiemCuoiKy
0,16.2,6.1,1,8.5,172,5.7
1,33.5,8.5,2,23.9,165,7.2
2,27.0,7.3,0,22.5,180,6.7
3,23.0,8.0,0,12.5,160,7.2
4,9.7,9.6,1,16.3,165,4.8
5,9.7,7.0,1,8.4,157,4.6
6,6.7,7.6,2,6.3,153,4.9
7,31.0,9.0,2,12.7,153,8.1
8,23.0,6.9,5,10.1,174,6.6
9,26.2,6.3,1,15.4,152,7.2


In [4]:
dataset.columns

Index(['SoGioHoc_Tuan', 'DiemChuyenCan', 'SoBuoiVang', 'SoGioChoiGame_Tuan',
       'ChieuCao_cm', 'DiemCuoiKy'],
      dtype='str')

In [2]:
feature_names = ['SoGioHoc_Tuan', 'DiemChuyenCan', 'SoBuoiVang', 'SoGioChoiGame_Tuan',
       'ChieuCao_cm']

X = dataset[feature_names]
y = dataset['DiemCuoiKy']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [3]:
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)
y_train_pred = lin_reg.predict(X_train)
y_test_pred = lin_reg.predict(X_test)

print(f'Hệ số chặn (Intercept w0): {lin_reg.intercept_:,.2f}')
for name, coef in zip(feature_names, lin_reg.coef_):
    print(f'Trọng số {name} (w): {coef:.4f}')

print('-' * 40)
print(f'R² Score trên Train Set: {r2_score(y_train, y_train_pred):.4f}')
print(f'RMSE trên Train Set:     ${root_mean_squared_error(y_train, y_train_pred):,.2f}')

print(f'R² Score trên Test Set: {r2_score(y_test, y_test_pred):.4f}')
print(f'RMSE trên Test Set:     ${root_mean_squared_error(y_test, y_test_pred):,.2f}')


Hệ số chặn (Intercept w0): 3.12
Trọng số SoGioHoc_Tuan (w): 0.1400
Trọng số DiemChuyenCan (w): 0.3766
Trọng số SoBuoiVang (w): -0.2677
Trọng số SoGioChoiGame_Tuan (w): -0.0839
Trọng số ChieuCao_cm (w): -0.0075
----------------------------------------
R² Score trên Train Set: 0.9030
RMSE trên Train Set:     $0.48
R² Score trên Test Set: 0.8222
RMSE trên Test Set:     $0.65


In [4]:
import numpy as np
import statsmodels.api as sm
# Thêm cột hệ số tự do w0 (toàn số 1: x0 = 1) vào ma trận X
X_opt = np.append(arr=np.ones((X.shape[0], 1)).astype(float), values=X, axis=1)
all_feature_names = ['Intercept (w0)'] + feature_names

# Huấn luyện mô hình OLS và truyền xname để hiển thị rõ tên từng đặc trưng
regressor_ols = sm.OLS(endog=y, exog=X_opt).fit()
print(regressor_ols.summary(xname=all_feature_names))


                            OLS Regression Results                            
Dep. Variable:             DiemCuoiKy   R-squared:                       0.890
Model:                            OLS   Adj. R-squared:                  0.884
Method:                 Least Squares   F-statistic:                     151.5
Date:                Fri, 02 Oct 2026   Prob (F-statistic):           2.19e-43
Time:                        20:45:41   Log-Likelihood:                -74.365
No. Observations:                 100   AIC:                             160.7
Df Residuals:                      94   BIC:                             176.4
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept (w0)         3.2790      0

In [5]:
import pandas as pd
import numpy as np
from IPython.display import display
from sklearn.linear_model import LinearRegression

# 1. Danh sách đầy đủ 5 đặc trưng (bao gồm cả 'ChieuCao_cm')
feature_names = ['SoGioHoc_Tuan', 'DiemChuyenCan', 'SoBuoiVang', 'SoGioChoiGame_Tuan', 'ChieuCao_cm']

# 2. Huấn luyện mô hình với đầy đủ 5 đặc trưng
lin_reg = LinearRegression()
lin_reg.fit(X_train[feature_names], y_train)

# 3. Chuẩn bị tập kiểm tra 5 đặc trưng
X_test_subset = X_test[feature_names]
X_test_arr = np.array(X_test_subset)
y_test_arr = np.array(y_test)

# Dự đoán điểm cuối kỳ trên tập test
y_pred = lin_reg.predict(X_test_subset)

# ==============================================================================
# 1. Xây dựng hàm compare() so sánh từng sinh viên riêng lẻ (đủ 5 đặc trưng)
# ==============================================================================
def compare(i_example):
    # Lấy mẫu sinh viên thứ i_example
    x = X_test_arr[i_example : i_example + 1]
    y_actual = y_test_arr[i_example]
    y_predicted = lin_reg.predict(X_test_subset.iloc[i_example : i_example + 1])[0]
    
    diff = y_actual - y_predicted
    error_pct = abs(diff) / y_actual * 100

    print(
        f"SV {i_example+1:2d} | "
        f"Giờ học: {x[0, 0]:4.1f}h/t | "
        f"Chuyên cần: {x[0, 1]:3.1f} đ | "
        f"Vắng: {int(x[0, 2]):1d} buổi | "
        f"Game: {x[0, 3]:4.1f}h/t | "
        f"Cao: {int(x[0, 4]):3d} cm | "
        f"Thực tế: {y_actual:4.1f} đ | "
        f"Dự đoán: {y_predicted:4.2f} đ | "
        f"Lệch: {diff:+5.2f} đ ({error_pct:5.2f}%)"
    )

print("=== Thử nghiệm hàm compare() với sinh viên đầu tiên trong tập Test ===")
compare(0)


# ==============================================================================
# 2. Lập bảng tổng hợp so sánh toàn bộ sinh viên trong tập Test (Đủ 5 đặc trưng)
# ==============================================================================
df_compare = pd.DataFrame({
    'STT': range(1, len(X_test_arr) + 1),
    'Giờ học/tuần': [f"{v:.1f}" for v in X_test_arr[:, 0]],
    'Điểm chuyên cần': [f"{v:.1f}" for v in X_test_arr[:, 1]],
    'Số buổi vắng': [int(v) for v in X_test_arr[:, 2]],
    'Giờ game/tuần': [f"{v:.1f}" for v in X_test_arr[:, 3]],
    'Chiều cao (cm)': [int(v) for v in X_test_arr[:, 4]],
    'Điểm thực tế': [f"{v:.1f}" for v in y_test_arr],
    'Điểm dự đoán': [f"{v:.2f}" for v in y_pred],
    'Chênh lệch': [f"{v:+.2f}" for v in (y_test_arr - y_pred)],
    'Sai số (%)': [f"{abs(act - pred)/act * 100:.2f}%" for act, pred in zip(y_test_arr, y_pred)]
})

print("\n=== Bảng Đối chiếu Toàn bộ Tập Test (Dự đoán Điểm Cuối Kỳ - 5 Đặc Trưng) ===")
display(df_compare)


=== Thử nghiệm hàm compare() với sinh viên đầu tiên trong tập Test ===
SV  1 | Giờ học:  6.9h/t | Chuyên cần: 9.5 đ | Vắng: 1 buổi | Game: 13.8h/t | Cao: 171 cm | Thực tế:  4.7 đ | Dự đoán: 4.97 đ | Lệch: -0.27 đ ( 5.69%)

=== Bảng Đối chiếu Toàn bộ Tập Test (Dự đoán Điểm Cuối Kỳ - 5 Đặc Trưng) ===


,STT,Giờ học/tuần,Điểm chuyên cần,Số buổi vắng,Giờ game/tuần,Chiều cao (cm),Điểm thực tế,Điểm dự đoán,Chênh lệch,Sai số (%)
0,1,6.9,9.5,1,13.8,171,4.7,4.97,-0.27,5.69%
1,2,31.8,8.0,2,23.9,152,5.7,6.91,-1.21,21.30%
2,3,28.2,8.7,3,2.7,159,8.5,8.13,+0.37,4.32%
3,4,24.9,6.1,0,24.9,177,6.1,5.50,+0.60,9.86%
4,5,12.8,7.1,1,25.0,175,4.3,3.92,+0.38,8.84%
5,6,18.2,9.9,4,11.5,157,6.1,6.19,-0.09,1.54%
6,7,13.8,7.3,5,18.7,155,5.1,3.74,+1.36,26.63%
7,8,30.9,7.4,1,12.1,183,8.3,7.59,+0.71,8.57%
8,9,5.6,7.2,2,3.8,181,5.0,4.42,+0.58,11.68%
9,10,16.2,6.1,1,8.5,172,5.7,5.43,+0.27,4.81%


Thay các giá trị trọng số coef vào, ta có:
   $$y = 3.2790 + 0.1378x_1 + 0.3155x_2 - 0.2388x_3 - 0.0865x_4 - 0.0052x_5$$

Chúng ta sẽ ráp các thông tin của An vào các biến tương ứng:
* $y$ (Điểm mục tiêu): $9.0$.   
* $x_2$ (Điểm chuyên cần): An không nghỉ buổi nào nên đạt $10.0$ điểm.   
* $x_3$ (Số buổi vắng): $0$.   
* $x_4$ (Số giờ chơi game/tuần): An cần thắng 10 trận, mỗi trận 40 phút. Do đó: $x_4 = 10 \times 40 \text{ phút} = 400 \text{ phút} \approx 6.6667 \text{ giờ}$.   
* $x_5$ (Chiều cao): 1m8 tức là $180$ cm.   
* $x_1$ (Số giờ học/tuần): Biến $x_1$ trong tập dữ liệu là số giờ học trung bình của cả học kỳ (18 tuần), chứ không phải chỉ riêng tuần cuối.   Trong 17 tuần trước, An học trung bình 28.0 giờ/tuần, tương đương: $17 \times 28 = 476 \text{ giờ}$.   Gọi số giờ An cần cày cuốc trong tuần cuối (tuần 18) là $H$.Vậy, số giờ học trung bình thực tế của An cho toàn học kỳ sẽ là: $x_1 = \frac{476 + H}{18}$.

$$9.0 = 3.2790 + 0.1378 \left( \frac{476 + H}{18} \right) + 0.3155(10.0) - 0.2388(0) - 0.0865(6.6667) - 0.0052(180)$$

$$\frac{476 + H}{18} = \frac{4.0797}{0.1378} \approx 29.6059$$

$$476 + H = 29.6059 \times 18 \approx 532.9062$$

$$H = 532.9062 - 476 = 56.9062 \text{ giờ}$$

Kết luận 1: An phải học trong 56.9 giờ thì mới kịp

Kết luận 2: trong tổng số 96 giờ của tuần 18 (từ thứ 2 đến thứ 6) thì An chỉ dùng 56.9 giờ và 6.667 giờ để chơi game thì vẫn còn hơn 32.43 giờ cho sinh hoạt, tương đương với 8.1 giờ/ngày cho sinh hoạt